# SEA Food Classifier — Best Config (yolo11s, full fine-tune)
**Before running:** Runtime → Change runtime type → **T4 GPU**.

Upload two files when prompted (or drop them in the Files sidebar first):
1. `food_yolo_classifier.zip` (the project)
2. `dataset.zip` (your images)

Expected total runtime on a T4: roughly 45–90 minutes. Target: **85%+ test accuracy**.

In [ ]:
# 1. Verify GPU
import torch
assert torch.cuda.is_available(), "No GPU! Runtime > Change runtime type > T4 GPU, then rerun."
print(torch.cuda.get_device_name(0))

In [ ]:
# 2. Install dependencies
!pip -q install ultralytics
# torch/torchvision/sklearn/matplotlib/pandas are preinstalled on Colab

In [ ]:
# 3. Upload project + dataset zips (skip if already in Files sidebar)
import os
from google.colab import files
if not os.path.exists('food_yolo_classifier.zip'):
    print('Select food_yolo_classifier.zip and dataset.zip')
    files.upload()

In [ ]:
# 4. Unpack
!unzip -qo food_yolo_classifier.zip
!unzip -qo dataset.zip -d food_yolo_classifier/ -x "__MACOSX/*"
!ls food_yolo_classifier/dataset

In [ ]:
# 5. Train (best config: yolo11s backbone, warmup + full fine-tune,
#    label smoothing, RandAugment, cosine LR, AMP)
%cd /content/food_yolo_classifier/src
!python train_best.py --data ../dataset --batch_size 64 --num_workers 2

In [ ]:
# 6. Full evaluation: precision / recall / F1 / confusion matrix
!python evaluate.py --data ../dataset \
    --ckpt ../checkpoints_best/best_model.pt \
    --out_dir ../results_best --unfreeze_last_n 11

**Note for step 6:** `evaluate.py` defaults to the yolo11n backbone. If it errors on a
shape mismatch, open `evaluate.py` and pass `weights_path='yolo11s.pt',
feature_channels=512` to `FoodYOLOClassifier` — or just use the test accuracy
already printed at the end of step 5 plus the cell below for the confusion matrix.

In [ ]:
# 6b. (fallback) metrics computed directly, guaranteed to match the checkpoint
import torch, numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import sys; sys.path.insert(0, '/content/food_yolo_classifier/src')
from model import FoodYOLOClassifier
from train_best import build_loaders, IMAGENET_MEAN, IMAGENET_STD

device = torch.device('cuda')
ck = torch.load('../checkpoints_best/best_model.pt', map_location=device)
classes = ck['classes']
model = FoodYOLOClassifier(num_classes=len(classes),
                           weights_path=ck.get('weights','yolo11s.pt'),
                           feature_channels=ck.get('feature_channels',512),
                           freeze_backbone=False).to(device)
model.load_state_dict(ck['model_state_dict']); model.eval()

_,_,test_loader,_ = build_loaders('../dataset', 64, 2)
yt, yp = [], []
with torch.no_grad():
    for x, y in test_loader:
        yp.extend(model(x.to(device)).argmax(1).cpu().tolist()); yt.extend(y.tolist())
print(classification_report(yt, yp, target_names=classes))

import matplotlib.pyplot as plt
cm = confusion_matrix(yt, yp)
fig, ax = plt.subplots(figsize=(8,7)); im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(classes))); ax.set_xticklabels(classes, rotation=45, ha='right')
ax.set_yticks(range(len(classes))); ax.set_yticklabels(classes)
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, cm[i,j], ha='center', va='center',
                color='white' if cm[i,j] > cm.max()/2 else 'black', fontsize=8)
plt.xlabel('Predicted'); plt.ylabel('True'); plt.tight_layout()
plt.savefig('../results_best/confusion_matrix.png', dpi=150); plt.show()

In [ ]:
# 7. Download the trained checkpoint + results to use in the Streamlit app
from google.colab import files
files.download('/content/food_yolo_classifier/checkpoints_best/best_model.pt')
files.download('/content/food_yolo_classifier/checkpoints_best/history.json')

## Using the new checkpoint in the Streamlit app
Copy `best_model.pt` into `checkpoints/` (replacing the old one), then in
`src/app.py` change the `load_model()` call to
`FoodYOLOClassifier(..., freeze_backbone=False)` and pass
`weights_path='yolo11s.pt', feature_channels=512`. Everything else is unchanged.

## Optional: even higher accuracy
- `--weights yolo11m.pt --feature_channels 576` (slower, may add ~1%)
- More real (non-augmented) images for the weakest classes from the
  confusion matrix — usually the noodle soups
- Train longer: raise `--finetune_epochs` to 60